# Step 6: pseudobulk (annotated version)

**The problem.** Question 2 needs a statistical test: which genes are higher in the population than in other
cells? Testing cell against cell would treat 50,799 population cells as 50,799 independent measurements. They
are not: they come from about a dozen embryos. Cells from one embryo share its genetics, age and handling.
Treating them as independent makes almost every gene look significant (this mistake is called
*pseudoreplication*).

**The solution: pseudobulk.** Add up the counts of all cells of one group from one donor into a single sample,
as if that group had been sequenced in bulk. Then the donors are the replicates, which is what they really are.
DESeq2 in step 07 is built for exactly this kind of data: raw counts, one sample per biological replicate.

**What one sample is:** all cells with the same **state**, **donor** and **kit**.

- *State* = `Population` for the 50,799 cells of the 8 clusters from step 05, otherwise the authors' cell
  class (Radial glia, Neuron, Glioblast, ...).
- Donor and kit are kept apart because each donor-kit pair is a separate library.

Example: all population cells of donor XDD:313 sequenced with v3 become the sample `Population|XDD:313|v3`.

Output for step 07: `data/processed/pseudobulk.npz`. No figure in this step.

*The output shown is from the run of 13 November 2026 on the full dataset. Running the notebook again gives
the identical file.*

## Setup

**What this cell does**

- Same first lines as every notebook: move to the repository root and allow imports from `src/`.
- **numpy** for the count tables, **pandas** for labels, **scipy.sparse** for the sparse matrix trick below.
- From `common.py`: dataset settings (`B`, for the gene symbol column), the config (`CFG`, for `chunk_size`),
  the `data/processed/` folder and the helpers to open the data and build the cell table.

In [1]:
# The notebooks live in notebooks/. Move to the repository root so config.yaml, data/ and results/
# are found, and make src/common.py importable.
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import numpy as np
import pandas as pd
import scipy.sparse as sp

from common import B, CFG, PROCESSED, cell_table, open_braun

## Load the data and label cell states

**What this cell does**

- Opens the h5ad and builds the per-cell table, as in every step.
- `population`: reads the 8 cluster IDs saved by step 05. `.read_text().split()` turns the file (one ID per
  line) into a list like `["150", "110", "145", ...]`.
- `state`: one label per cell.
  - `cells["cluster"].isin(population)` is True for cells in the 8 clusters
  - `cells["cell_class"].where(condition, "Population")` keeps the cell class where the condition is True and
    puts `"Population"` everywhere else. Here the condition is `~...isin(...)` ("not in the population"), so:
    population cells get `"Population"`, all other cells keep their class

Note: the 8 clusters are radial glia, so after this step the state `Radial glia` means *radial glia that are
not in the population*. That matters in step 07, where late radial glia are the reference on the stage axis.

In [2]:
adata = open_braun()
cells = cell_table(adata)
# Cells in the population clusters get the label "Population"; all others keep their cell class
population = (PROCESSED / "population_clusters.txt").read_text().split()
state = cells["cell_class"].where(~cells["cluster"].isin(population), "Population")

## Sum the counts per state, donor and kit

**Step 1, give every cell a sample number**

- `state + "|" + donor + "|" + chemistry` builds a label like `"Population|XDD:313|v3"` for every cell.
- `pd.factorize(...)` turns the labels into numbers: `sample_of_cell` holds the sample number of each cell
  (0, 1, 2, ...) and `samples` holds the label of each number.
- `counts` and `expressing`: empty tables with one row per sample and one column per gene (59,459), filled by
  the loop.

**Step 2, read the matrix in blocks and add up**

The matrix is too big for memory, so the loop reads `chunk_size` (20,000) cells at a time. For each block:

- `X`: the block's counts (sparse, only non-zero values stored).
- `s`: the sample number of each cell in the block.
- `member`: a small matrix with one row per sample and one column per cell in the block, with a 1 where the
  cell belongs to the sample.
- `member @ X`: matrix multiplication. Each sample row picks up and adds the counts of its own cells.

**Toy example:** 3 cells, 2 genes, 2 samples (cells 1 and 3 belong to sample A, cell 2 to sample B).

| | gene 1 | gene 2 |
|---|---|---|
| cell 1 | 2 | 0 |
| cell 2 | 1 | 4 |
| cell 3 | 3 | 1 |

`member` (rows = samples, columns = cells):

| | cell 1 | cell 2 | cell 3 |
|---|---|---|---|
| sample A | 1 | 0 | 1 |
| sample B | 0 | 1 | 0 |

`member @ X`: sample A = cell 1 + cell 3 = (2+3, 0+1) = **(5, 1)**; sample B = cell 2 = **(1, 4)**.
That is the pseudobulk: one summed row per sample. With 20,000 cells and 59,459 genes the same multiplication
does millions of additions in one step, much faster than looping over cells.

**Step 3, count expressing cells with the same trick**

- `X.data[:] = 1` replaces every non-zero count by 1, so the block now says "expressed or not" (zeros are not
  stored, so they stay 0).
- `member @ X` now adds up ones: for each sample and gene, the **number of cells that express the gene**.
- Step 07 divides this by the number of cells to get the `% cells expressing` columns in the marker tables, an
  easy-to-read effect size next to the fold change.

In the toy example, after setting non-zeros to 1: sample A, gene 2 = 0 + 1 = **1 cell** expresses it (cell 3),
gene 1 = **2 cells**.

This cell reads the whole matrix once and takes a few minutes. It prints nothing.

In [3]:
# One pseudobulk sample per state x donor x kit, e.g. "Population|XDD:313|v3"
sample_of_cell, samples = pd.factorize(state + "|" + cells["donor"] + "|" + cells["chemistry"])
counts = np.zeros((len(samples), adata.n_vars))
expressing = np.zeros((len(samples), adata.n_vars))  # cells with the gene > 0, for % expressing

# Read the matrix in blocks of cells; the whole matrix would not fit in memory
for start in range(0, adata.n_obs, CFG["chunk_size"]):
    X = adata.X[start:start + CFG["chunk_size"]]
    s = sample_of_cell[start:start + CFG["chunk_size"]]
    # samples x cells matrix with a 1 where a cell belongs to a sample; multiplying sums the cells
    member = sp.csr_matrix((np.ones(len(s)), (s, np.arange(len(s)))), shape=(len(samples), len(s)))
    counts += (member @ X).toarray()
    # Set every non-zero count to 1: the same multiplication now counts expressing cells
    X.data[:] = 1
    expressing += (member @ X).toarray()

## Save the pseudobulk samples

**What this cell does**

- `info`: a table describing each sample. `s.split("|")` splits each label back into state, donor and kit.
- `info["age_bin"]`: each donor has one age, so its age bin is looked up from the cell table
  (`groupby("donor")["age_bin"].first()` gives one age bin per donor; `.map` fills it in for each sample).
- `np.savez_compressed(...)` stores everything step 07 needs in one compressed file:

| Name | Content |
|---|---|
| `counts` | summed counts, samples x genes (whole numbers, as DESeq2 needs) |
| `expressing` | number of expressing cells, samples x genes |
| `n_cells` | how many cells went into each sample (`np.bincount` counts how often each sample number occurs) |
| `genes`, `ensembl` | gene symbols and Ensembl IDs, the column labels |
| `state`, `donor`, `chemistry`, `age_bin` | the sample table |

Because everything is in this one file, step 07 never has to open the 12 GB h5ad.

In [4]:
# Sample table: state, donor, kit and the donor's age bin (one donor has one age)
info = pd.DataFrame([s.split("|") for s in samples], columns=["state", "donor", "chemistry"])
info["age_bin"] = info["donor"].map(cells.groupby("donor")["age_bin"].first())
# One compressed file with everything step 7 needs, so step 7 never has to open the 12 GB h5ad
np.savez_compressed(PROCESSED / "pseudobulk.npz", counts=counts.astype(np.int64),
                    expressing=expressing.astype(np.int64), n_cells=np.bincount(sample_of_cell),
                    genes=adata.var[B["symbol_col"]].to_numpy(), ensembl=adata.var_names.to_numpy(),
                    **{col: info[col].to_numpy() for col in info})
print(f"{len(samples)} pseudobulk samples")

299 pseudobulk samples


**Reading the output:** 299 samples, one for every state, donor and kit combination that has at least
one cell. Not all of them are used in step 07: there, samples are pooled per comparison and samples built from
fewer than 50 cells are dropped, because their counts are too noisy.

## Conclusion of step 06

1. 1,665,937 cells are reduced to 299 pseudobulk samples (state x donor x kit), so donors become the
   replicates and the tests in step 07 are not inflated by treating cells as independent.
2. Each sample holds raw summed counts, the input DESeq2 needs, plus the number of expressing cells per gene
   for the `% cells` columns.
3. Population cells form their own state. The remaining radial glia form the state `Radial glia`.